# Shark survey batch pipeline

Runs the full chain for one flight **sequence** at a time (set `CURRENT_SEQUENCE` in the
"Run" cell below), picked from every sequence listed in `shark-data-info.ods`:

```
SRT + AirData + video clips
        |  generate_flat_surface_dem.py         (per-sequence flat DEM, sea level)
        v
  flat_surface_dem.json
        |  extract_video_frames.py               (frames + poses.json, no QGIS needed)
        v
   frames_w/poses.json
        |  trex_to_bambi.py                       (geo-reference the TRex tracklets)
        v
   tracks_w/tracks.csv  (+ detections_w, tracks_pixel_w, georeferenced_w)
```

Assumptions baked into this notebook (change in the config cell if wrong):

- All flights happened in the Maldives -> timezone `Indian/Maldives` for every sequence.
- No real bathymetric DEM is available for these sites -> every sequence uses a flat
  sea-level surface (`--flat-surface-msl 0.0`) generated fresh per sequence from its own
  GPS track (`generate_flat_surface_dem.py`).
- Camera is always the RGB/wide (`W`) camera.
- Drone (`pink`/`blue`) — and therefore which camera calibration to use — is inferred from
  which subfolder the sequence's AirData CSV lives in
  (`log-files/airdata/pink/...` vs `log-files/airdata/blue/...`).
- TRex tracklet variant: always the plain `*_id*.npz` files for sharks. `*_posture_id*.npz`
  is a *different* TRex export type (posture/midline shape analysis only) that doesn't
  carry the frame/pose-keypoint/detection fields the shark pipeline needs, so it's never
  used as a shark tracklet. Fish-school detections (`*_fschool_id*.npz` /
  `*_fschool_posture_id*.npz`) are a separate track kind, handled by their own step
  (Step 3c) using the posture file's outline/hole polygons instead of pose key-points.
- A sequence is only processed if its `airdata-file` column is filled in in the spreadsheet.
  The ordered list of video clips is taken from `<sequence>.txt` (an ffmpeg-concat-style list
  of clip filenames) if that file exists next to the SRTs, otherwise the sequence is assumed
  to be a single clip named `<sequence>.MP4`.

Everything is idempotent: re-running the "Run" cell for the same sequence skips whichever
stages already have output.

In [5]:
import os

# cv2/numpy/BLAS default to spawning up to nproc threads, often with busy-spinning
# idle worker threads once initialized. This env block MUST run before pandas/numpy
# are imported (their thread pools latch onto these values on first use) - otherwise
# even this notebook's own kernel process ends up with a permanently bloated thread
# pool for its whole session, starving every subprocess we launch afterwards. If
# you've already run this notebook once without this cap, RESTART THE KERNEL (not
# just re-run cells) - an already-initialized thread pool can't be shrunk in-process.
THREAD_LIMIT = "4"
os.environ["OMP_NUM_THREADS"] = THREAD_LIMIT
os.environ["OPENBLAS_NUM_THREADS"] = THREAD_LIMIT
os.environ["MKL_NUM_THREADS"] = THREAD_LIMIT
os.environ["NUMEXPR_NUM_THREADS"] = THREAD_LIMIT
os.environ["OPENCV_NUM_THREADS"] = THREAD_LIMIT

import json
import re
import shutil
import subprocess
import sys

import cv2
import pandas as pd

# ── Repos ────────────────────────────────────────────────────────────────────
BAMBI_REPO = "/home/angela/Seafile/test-data/pink/code/bambi_detection"
TREX_REPO = "/home/angela/Seafile/test-data/pink/code/TRexConnector"

EXTRACT_SCRIPT = os.path.join(BAMBI_REPO, "src", "bambi", "extract_video_frames.py")
DEM_SCRIPT = os.path.join(BAMBI_REPO, "src", "bambi", "generate_flat_surface_dem.py")
TREX_SCRIPT = os.path.join(TREX_REPO, "trex_to_bambi.py")

# trex_to_bambi.py needs alfspy/bambi, which aren't installed in this notebook's kernel
# env - run it with the dedicated "trexconnector" conda env instead of sys.executable.
TREX_PYTHON = "/home/angela/miniforge3/envs/trexconnector/bin/python"

# ── Raw data ─────────────────────────────────────────────────────────────────
SHARKS_ROOT = "/media/angela/2026-drone/sharks_data"
VIDEOS_ROOT = "/media/maldives/2024_sequences/sequences-yolo26"
NPZ_DIR = os.path.join(VIDEOS_ROOT, "data")
SRT_DIR = os.path.join(SHARKS_ROOT, "log-files", "SRT")
CALIB_DIR = os.path.join(SHARKS_ROOT, "camera-calibration-data")
ODS_PATH = os.path.join(SHARKS_ROOT, "hand-timestamps", "shark-data-info.ods")

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_ROOT = os.path.join(SHARKS_ROOT, "processed_data", "bambi_pipeline")
GEOREF_VIDEOS_DIR = os.path.join(SHARKS_ROOT, "georeferenced-videos")

# ── Fixed assumptions (see markdown above) ──────────────────────────────────
TIMEZONE = "Indian/Maldives"
CAMERA = "W"
FLAT_SURFACE_MSL = 0.0

# ── Debug mode ───────────────────────────────────────────────────────────────
# Set to an int (e.g. 200) to only extract/georeference/render the first N frames of
# the sequence being run - useful for a quick end-to-end test before committing to a
# full run. Debug output goes to a separate "<out_dir>_debug<N>" tree so it never
# collides with (or gets mistaken for) real full-run output. Set back to None for
# full runs. Requires the TimedFrameExtractorCallback early-stop fix in bambi's
# webgl/timed_pose_extractor.py - otherwise --limit doesn't actually skip decoding.
DEBUG_FRAME_LIMIT = None

def entry_out_dir(entry):
    """Per-sequence output dir, redirected to a debug-specific tree when
    DEBUG_FRAME_LIMIT is set (see 'Debug mode' above)."""
    if DEBUG_FRAME_LIMIT is None:
        return entry["out_dir"]
    return f"{entry['out_dir']}_debug{DEBUG_FRAME_LIMIT}"


def frames_dir_for(entry):
    """Deterministic frames_w dir for a sequence. Computed from the entry itself
    rather than read back from entry["frames_dir"], so later stages don't depend on
    the extraction stage having run for this entry in the *current* kernel session -
    they can be (re-)run standalone, e.g. after a kernel restart."""
    return os.path.join(entry_out_dir(entry), "frames_w")


def _poses_frame_count(poses_json_path):
    """Number of frames actually recorded in a poses.json (0 if missing or
    unreadable - e.g. a previous extraction that was interrupted mid-write,
    leaving a truncated or empty file behind)."""
    if not os.path.isfile(poses_json_path):
        return 0
    try:
        with open(poses_json_path, "r", encoding="utf-8") as f:
            poses = json.load(f)
    except (json.JSONDecodeError, OSError) as exc:
        print(f"[warn] {poses_json_path}: unreadable ({exc}) - treating as incomplete")
        return 0
    return len(poses["images"] if "images" in poses else poses)


def _expected_frame_count(entry):
    """Total frames expected across this sequence's video clip(s) (concatenated),
    capped by DEBUG_FRAME_LIMIT when set - that's an intentional partial extraction,
    not an incomplete one."""
    total = 0
    for video_path in entry["videos"]:
        cap = cv2.VideoCapture(video_path)
        total += int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
    if DEBUG_FRAME_LIMIT is not None:
        total = min(total, DEBUG_FRAME_LIMIT)
    return total


def frames_ready(entry):
    """Whether frame extraction has *fully* finished for this entry: poses.json
    exists AND its frame count meets the expected total (see _expected_frame_count).
    Just checking "poses.json exists" would also be true for an interrupted or
    crashed extraction that stopped partway through with fewer frames than
    expected - this catches that instead of silently treating it as done."""
    n_actual = _poses_frame_count(os.path.join(frames_dir_for(entry), "poses.json"))
    if n_actual == 0:
        return False
    n_expected = _expected_frame_count(entry)
    return n_expected > 0 and n_actual >= n_expected


def run_capped(cmd):
    """subprocess.run with the thread cap above and check=True."""
    subprocess.run(cmd, check=True, env=os.environ)


os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(GEOREF_VIDEOS_DIR, exist_ok=True)

## Step 0 — Build the manifest

Reads the `overview` sheet, keeps sequences that have an `airdata-file`, and resolves every
path the rest of the pipeline needs. Nothing is executed on the *video/tracking* data yet -
this only inspects the filesystem so problems (missing clips, no matching tracklets, ...)
show up before any processing starts.

One exception: for multi-clip sequences that only have a pre-concatenated single video left
on disk (the 9 raw per-clip files having been deleted), DJI still only wrote one `.SRT` per
raw clip, while `extract_video_frames.py` requires exactly one SRT per video file. So this
step builds (and caches next to the raw SRTs, as `<sequence>_merged.SRT`) a single merged SRT
from the raw per-clip SRTs, using `bambi.srt.SrtMerger`. This was verified to line up exactly,
frame-for-frame, with the merged video (merged SRT frame count == video frame count, and the
embedded telemetry timestamps are continuous with no gap across clip boundaries).

In [6]:
def _parse_concat_list(path):
    """Parse an ffmpeg-concat-style 'file <name>' list into an ordered filename list."""
    names = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("file "):
                names.append(line[len("file "):].strip().strip("'\""))
    return names


def _resolve_drone(airdata_path):
    if f"{os.sep}airdata{os.sep}pink{os.sep}" in airdata_path:
        return "pink"
    if f"{os.sep}airdata{os.sep}blue{os.sep}" in airdata_path:
        return "blue"
    return None


def _resolve_npz(seq):
    """Pick the tracklet npz files for a sequence: the plain *_id*.npz files, always
    excluding anything with 'fschool' in the name.

    *_posture_id*.npz is a *different* TRex export type (posture/midline shape
    analysis only - fields like "frames", "midline_lengths", "hole_points") and does
    NOT carry the frame/pose-keypoint/detection_class fields trex_to_bambi.py needs -
    it is not a richer/preferred version of the plain tracklet data, so it must never
    be used as the tracklet source here.
    """
    if not os.path.isdir(NPZ_DIR):
        return None, []
    candidates = [
        f for f in os.listdir(NPZ_DIR)
        if f.startswith(seq + "_") and f.endswith(".npz")
        and "fschool" not in f and "_posture_id" not in f
    ]
    if candidates:
        return "plain", sorted(candidates)
    return None, []


def _resolve_fschool(seq):
    """Fish-school posture tracklet ids for a sequence: every id with a
    <seq>_fschool_posture_id<N>.npz file (the plain <seq>_fschool_id<N>.npz is a
    simple bbox/centroid track like the shark npz - only the posture file carries
    the outline/hole polygons export_georeferenced_fschool.py and the visualizer
    actually use, so that's what gates whether an id is usable here)."""
    if not os.path.isdir(NPZ_DIR):
        return []
    ids = []
    for f in os.listdir(NPZ_DIR):
        if f.startswith(f"{seq}_fschool_posture_id") and f.endswith(".npz"):
            m = re.search(r"_id(\d+)\.npz$", f)
            if m:
                ids.append(int(m.group(1)))
    return sorted(ids)


def _build_merged_srt(seq, raw_srt_names):
    """Merge a multi-clip sequence's raw per-clip SRTs into one SRT matching the
    already-concatenated video. extract_video_frames.py requires exactly one SRT per
    video file; DJI writes one SRT per raw clip instead. Cached next to the raw SRTs
    as "<sequence>_merged.SRT" (all drones/cameras in this survey are DJI M30T Wide).
    """
    from bambi.domain.camera import Camera
    from bambi.domain.drone import Drone
    from bambi.srt.srt_merger import SrtMerger
    from bambi.srt.srt_parser import SrtParser
    from bambi.srt.srt_writer import SrtWriter

    merged_path = os.path.join(SRT_DIR, f"{seq}_merged.SRT")
    if os.path.isfile(merged_path):
        return merged_path

    parser = SrtParser()
    frame_lists = [parser.parse(os.path.join(SRT_DIR, name)) for name in raw_srt_names]
    merged_frames = SrtMerger().merge(frame_lists)
    SrtWriter().write(merged_path, merged_frames, Drone.M30T, Camera.Wide)
    print(f"[run]  {seq}: merged {len(raw_srt_names)} per-clip SRTs -> {os.path.basename(merged_path)}")
    return merged_path


def _resolve_video_and_srts(seq):
    """Pick the video file(s) and matching SRT file(s) for a sequence.

    Prefer an already-concatenated single video matching the sequence name exactly
    (e.g. "sequence_<id>.mp4") over the raw per-clip files listed in the sequence's
    concat .txt list: TRex was run directly against that merged file (confirmed via
    its .settings "meta_source_path"), and for many sequences the raw per-clip files
    no longer exist on disk at all - only the merged file remains.

    When using the merged video and the concat .txt lists more than one raw clip, the
    matching SRT is built via _build_merged_srt from those clips' raw per-clip SRTs.
    Otherwise (a genuinely single-clip sequence, or no concat .txt) falls back to the
    plain per-clip-name-based SRT guess - no merge needed for a single file.
    """
    concat_txt = os.path.join(SRT_DIR, f"{seq}.txt")
    raw_clip_names = _parse_concat_list(concat_txt) if os.path.isfile(concat_txt) else None

    for ext in (".mp4", ".MP4"):
        single_video = os.path.join(VIDEOS_ROOT, f"{seq}{ext}")
        if os.path.isfile(single_video):
            videos = [single_video]
            if raw_clip_names and len(raw_clip_names) > 1:
                raw_srt_names = [os.path.splitext(n)[0] + ".SRT" for n in raw_clip_names]
                if all(os.path.isfile(os.path.join(SRT_DIR, n)) for n in raw_srt_names):
                    return videos, [_build_merged_srt(seq, raw_srt_names)]
            srt_guess = os.path.join(SRT_DIR, os.path.splitext(os.path.basename(single_video))[0] + ".SRT")
            return videos, [srt_guess]

    clip_names = raw_clip_names or [f"{seq}.MP4"]
    videos = [os.path.join(VIDEOS_ROOT, name) for name in clip_names]
    srts = [os.path.join(SRT_DIR, os.path.splitext(name)[0] + ".SRT") for name in clip_names]
    return videos, srts


def resolve_sequence(row):
    seq = row["Sequence"]
    airdata = row["airdata-file"]
    if pd.isna(seq) or pd.isna(airdata):
        return None

    drone = _resolve_drone(airdata)
    videos, srts = _resolve_video_and_srts(seq)
    calib_path = os.path.join(CALIB_DIR, f"{drone}_drone_combined.json") if drone else None
    npz_variant, npz_names = _resolve_npz(seq)
    fschool_ids = _resolve_fschool(seq)

    missing_videos = [v for v in videos if not os.path.isfile(v)]
    missing_srts = [s for s in srts if not os.path.isfile(s)]
    missing_reasons = []
    if drone is None:
        missing_reasons.append("unrecognised drone (airdata not under pink/ or blue/)")
    if missing_videos:
        missing_reasons.append(f"{len(missing_videos)} missing video clip(s)")
    if missing_srts:
        missing_reasons.append(f"{len(missing_srts)} missing SRT file(s)")
    if npz_variant is None:
        missing_reasons.append("no matching (non-fschool) TRex tracklets")
    if calib_path and not os.path.isfile(calib_path):
        missing_reasons.append(f"calibration file not found: {calib_path}")

    return {
        "sequence": seq,
        "drone": drone,
        "airdata": airdata,
        "videos": videos,
        "srts": srts,
        "num_clips": len(videos),
        "calib_path": calib_path,
        "npz_variant": npz_variant,
        "npz_paths": [os.path.join(NPZ_DIR, n) for n in npz_names],
        "fschool_ids": fschool_ids,
        "ready": not missing_reasons,
        "issues": "; ".join(missing_reasons),
        "out_dir": os.path.join(OUTPUT_ROOT, seq),
    }


overview = pd.read_excel(ODS_PATH, engine="odf", sheet_name="overview")
manifest = [resolve_sequence(row) for _, row in overview.iterrows()]
manifest = [m for m in manifest if m is not None]
print(f"{len(manifest)} sequences have an airdata-file; "
      f"{sum(m['ready'] for m in manifest)} are ready to process.")

11 sequences have an airdata-file; 11 are ready to process.


In [7]:
_manifest_df = pd.DataFrame(manifest)
_manifest_df["n_fschool"] = _manifest_df["fschool_ids"].apply(len)
_manifest_df[["sequence", "drone", "num_clips", "npz_variant", "n_fschool", "ready", "issues"]]

,sequence,drone,num_clips,npz_variant,n_fschool,ready,issues
0,sequence_20240303_070126703_DJI_0257,blue,1,plain,2,True,
1,sequence_20240303_060726300_DJI_0242,blue,1,plain,3,True,
2,sequence_20240303_072831125_DJI_0266,blue,1,plain,3,True,
3,sequence_20240305_070907124_DJI_0266,pink,1,plain,3,True,
4,sequence_20240305_164039273_DJI_0305,pink,1,plain,2,True,
5,sequence_20240306_161443984_DJI_0139,blue,1,plain,2,True,
6,sequence_20240305_063615511_DJI_0253,pink,1,plain,3,True,
7,sequence_20240308_074843321_DJI_0502,blue,1,plain,2,True,
8,sequence_20240302_083729249_DJI_0001,pink,1,plain,1,True,
9,sequence_20240307_070355688_DJI_0202,blue,1,plain,1,True,


## Step 1 — Generate a flat-surface DEM per sequence

Uses the sequence's own AirData CSV to compute a GPS-centred flat surface at sea level
(`--elevation 0.0`); the UTM zone is auto-detected per sequence. Skips sequences that already
have a `flat_surface_dem.json`.

In [8]:
def generate_dem(entry):
    dem_dir = os.path.join(entry["out_dir"], "dem")
    dem_json = os.path.join(dem_dir, "flat_surface_dem.json")
    if os.path.isfile(dem_json):
        print(f"[skip] {entry['sequence']}: DEM already exists")
        return dem_json
    os.makedirs(dem_dir, exist_ok=True)
    cmd = [
        sys.executable, DEM_SCRIPT,
        "--inputs", entry["airdata"],
        "--elevation", str(FLAT_SURFACE_MSL),
        "--output", dem_dir,
        "--name", "flat_surface_dem",
    ]
    print(f"[run]  {entry['sequence']}: generating flat-surface DEM")
    run_capped(cmd)
    return dem_json

## Step 2 — Extract frames + poses

Runs `extract_video_frames.py` for each sequence (all its clips, in concat order), anchored
to the DEM origin from step 1. Skips sequences that already have a `poses.json`.

In [9]:
def extract_frames(entry):
    frames_dir = frames_dir_for(entry)
    poses_json = os.path.join(frames_dir, "poses.json")
    if frames_ready(entry):
        print(f"[skip] {entry['sequence']}: frames already extracted "
              f"({_poses_frame_count(poses_json)} frames)")
        return frames_dir
    n_actual = _poses_frame_count(poses_json)
    if n_actual:
        print(f"[redo] {entry['sequence']}: incomplete extraction found "
              f"({n_actual}/{_expected_frame_count(entry)} frames) - re-running")
    cmd = [
        sys.executable, EXTRACT_SCRIPT,
        "--videos", *entry["videos"],
        "--srts", *entry["srts"],
        "--airdata", entry["airdata"],
        "--camera", CAMERA,
        "--calibration-path", entry["calib_path"],
        "--dem-json", entry["dem_json"],
        "--output", frames_dir,
        "--timezone", TIMEZONE,
    ]
    if DEBUG_FRAME_LIMIT is not None:
        cmd += ["--limit", str(DEBUG_FRAME_LIMIT)]
    print(f"[run]  {entry['sequence']}: extracting {entry['num_clips']} clip(s)"
          + (f" (debug: first {DEBUG_FRAME_LIMIT} frames)" if DEBUG_FRAME_LIMIT is not None else ""))
    run_capped(cmd)
    return frames_dir

## Step 3 — Geo-reference the TRex tracklets

The plain tracklets (`*_id*.npz`, never `_posture_id` or `fschool`) are copied into a
per-sequence staging folder, since `trex_to_bambi.py` reads *every* `.npz` in its
`--npz-dir` and the real tracklets for all sequences live together in one shared folder.
(Copied rather than symlinked because the output drive is exFAT, which doesn't support
symlinks.) Skips sequences that already have a `tracks.csv`.

In [10]:
def stage_npz(entry):
    """Copy this sequence's tracklet npz files into a per-sequence staging folder
    (trex_to_bambi.py reads *every* .npz in --npz-dir). Syncs rather than just
    adding: also removes any stale .npz left over from a previous run whose file set
    doesn't match the current entry["npz_paths"] (e.g. after _resolve_npz's logic
    changed, or a debug re-run) - trex_to_bambi.py has no way to filter unwanted
    files itself, so a stale leftover silently corrupts the read.
    """
    staging = os.path.join(entry_out_dir(entry), "npz_input")
    os.makedirs(staging, exist_ok=True)
    wanted = {os.path.basename(src) for src in entry["npz_paths"]}
    for existing in os.listdir(staging):
        if existing.endswith(".npz") and existing not in wanted:
            os.remove(os.path.join(staging, existing))
    for src in entry["npz_paths"]:
        dst = os.path.join(staging, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    return staging


def run_trex_connector(entry):
    out_dir = entry_out_dir(entry)
    tracks_csv = os.path.join(out_dir, "tracks_w", "tracks.csv")
    if os.path.isfile(tracks_csv):
        print(f"[skip] {entry['sequence']}: already geo-referenced")
        return
    frames_dir = frames_dir_for(entry)
    npz_dir = stage_npz(entry)
    mask_path = os.path.join(frames_dir, f"mask_{CAMERA}.png")
    cmd = [
        TREX_PYTHON, TREX_SCRIPT,
        "--npz-dir", npz_dir,
        "--dem-json", entry["dem_json"],
        "--poses", os.path.join(frames_dir, "poses.json"),
        "--calib", entry["calib_path"],
        "--mask", mask_path,
        "--out-dir", out_dir,
        "--flat-surface-msl", str(FLAT_SURFACE_MSL),
    ]
    print(f"[run]  {entry['sequence']}: geo-referencing ({entry['npz_variant']} tracklets)")
    run_capped(cmd)

## Step 3b — Export geo-referenced tracklets (per key-point)

Step 3 only geo-references the *bounding box* of each TRex detection. This step goes
further: it re-reads the same staged `npz_input/*.npz` tracklets and writes an augmented
copy of each one, keeping every original field untouched (`frame`, `timestamp`,
`detection_p`, `id`, ...) and adding a `poseX{i}_geor` / `poseY{i}_geor` pair for every
existing `poseX{i}`/`poseY{i}` key-point - the world-space (DEM CRS) x/y of that
key-point, using the same undistort + DEM/flat-surface projection as Step 3, applied
per key-point instead of just the 4 bbox corners. Key-points that fail to project (frame
beyond the poses range, or the ray misses the projection surface - roughly half of them,
same failure rate as Step 3's bounding boxes) are left as `NaN`.

Output: one npz per input tracklet (same filename), written to
`tracklets_georeferenced_w/`. Skips sequences whose output already has one npz per input
tracklet. This is ray-cast bound and can take ~2 minutes per sequence; independent of
Step 4 (rendering).

In [11]:
EXPORT_TRACKLETS_SCRIPT = os.path.join(TREX_REPO, "export_georeferenced_tracklets.py")


def export_georeferenced_tracklets(entry):
    out_dir = os.path.join(entry_out_dir(entry), "tracklets_georeferenced_w")
    npz_dir = stage_npz(entry)  # reuse Step 3's staging (also syncs stale files)
    wanted = {os.path.basename(p) for p in entry["npz_paths"]}
    if os.path.isdir(out_dir):
        existing = {f for f in os.listdir(out_dir) if f.endswith(".npz")}
        if wanted and wanted.issubset(existing):
            print(f"[skip] {entry['sequence']}: geo-referenced tracklets already exported")
            return out_dir
    frames_dir = frames_dir_for(entry)
    cmd = [
        TREX_PYTHON, EXPORT_TRACKLETS_SCRIPT,
        "--npz-dir", npz_dir,
        "--dem-json", entry["dem_json"],
        "--poses", os.path.join(frames_dir, "poses.json"),
        "--calib", entry["calib_path"],
        "--mask", os.path.join(frames_dir, f"mask_{CAMERA}.png"),
        "--out-dir", out_dir,
        "--flat-surface-msl", str(FLAT_SURFACE_MSL),
    ]
    print(f"[run]  {entry['sequence']}: exporting geo-referenced tracklets ({len(wanted)} tracks)")
    run_capped(cmd)
    return out_dir

## Step 3c — Geo-reference fish-school outlines/holes

Fish-school detections are a different TRex export (`*_fschool_posture_id<N>.npz`) from
the shark tracklets: instead of a fixed 9 pose key-points, each frame carries a
variable-length outline polygon plus zero or more "hole" polygons (gaps inside the school
shape), stored as raw per-pixel contour traces (thousands of points/frame - see
`fschool_posture.py`). `export_georeferenced_fschool.py` decimates every polygon to
`--max-polygon-points` and geo-references it with the same undistort + flat-surface
projection as Steps 3/3b, adding `outline_points_geor`/`hole_points_geor` (+ their own
decimated length arrays) alongside the untouched original fields.

Output: one npz per fish-school track (same filename), written to
`fschool_georeferenced_w/`. Skips sequences with no fish-school tracklets, and skips
tracks whose output already exists. Ray-cast bound, similar cost per track to Step 3b.

In [12]:
FSCHOOL_EXPORT_SCRIPT = os.path.join(TREX_REPO, "export_georeferenced_fschool.py")


def georeference_fschool(entry):
    if not entry["fschool_ids"]:
        print(f"[skip] {entry['sequence']}: no fish-school posture tracklets")
        return None
    out_dir = os.path.join(entry_out_dir(entry), "fschool_georeferenced_w")
    wanted = {f"{entry['sequence']}_fschool_posture_id{i}.npz" for i in entry["fschool_ids"]}
    if os.path.isdir(out_dir):
        existing = {f for f in os.listdir(out_dir) if f.endswith(".npz")}
        if wanted.issubset(existing):
            print(f"[skip] {entry['sequence']}: fish-school outlines already geo-referenced")
            return out_dir
    frames_dir = frames_dir_for(entry)
    cmd = [
        TREX_PYTHON, FSCHOOL_EXPORT_SCRIPT,
        "--npz-dir", NPZ_DIR,
        "--sequence", entry["sequence"],
        "--video", entry["videos"][0],
        "--dem-json", entry["dem_json"],
        "--poses", os.path.join(frames_dir, "poses.json"),
        "--calib", entry["calib_path"],
        "--mask", os.path.join(frames_dir, f"mask_{CAMERA}.png"),
        "--out-dir", out_dir,
        "--flat-surface-msl", str(FLAT_SURFACE_MSL),
    ]
    print(f"[run]  {entry['sequence']}: geo-referencing {len(wanted)} fish-school track(s)")
    run_capped(cmd)
    return out_dir

## Step 4 — Render georeferenced videos

Renders the side-by-side pixel-space / geo-space video for every `ready` sequence (full
length, no frame cap) and writes each one to `GEOREF_VIDEOS_DIR` as
`<sequence>_trex_vis.mp4`. Skips sequences whose output video already exists.

Note: `--calib` is intentionally *not* passed here — `--video` is the original raw video,
and `--calib` would undistort the keypoints into a space the raw video isn't in, misaligning
them. Only use `--calib` when `--video` points at the undistorted extracted frames instead.

In [13]:

# Rendering the full-length side-by-side video for a sequence is slow, so by default
# only a window of the video is rendered. Set RENDER_MAX_FRAMES to None (and leave
# start/end at their defaults) to render the whole sequence instead.
RENDER_START_FRAME = 0
RENDER_END_FRAME = None  # exclusive; overrides RENDER_MAX_FRAMES if both are set
RENDER_MAX_FRAMES = 3000  # frames counted from RENDER_START_FRAME; set to None for no cap


def render_georeferenced_video(entry):
    # Encode the rendered frame range in the filename (when not a full-length render)
    # so different RENDER_START_FRAME/RENDER_END_FRAME/RENDER_MAX_FRAMES settings don't
    # collide with each other or get mistaken for a full-length render.
    range_suffix = ""
    if RENDER_START_FRAME or RENDER_END_FRAME is not None or RENDER_MAX_FRAMES is not None:
        if RENDER_END_FRAME is not None:
            end_label = RENDER_END_FRAME
        elif RENDER_MAX_FRAMES is not None:
            end_label = RENDER_START_FRAME + RENDER_MAX_FRAMES
        else:
            end_label = "end"
        range_suffix = f"_f{RENDER_START_FRAME}-{end_label}"
    suffix = f"_debug{DEBUG_FRAME_LIMIT}" if DEBUG_FRAME_LIMIT is not None else ""
    out_path = os.path.join(GEOREF_VIDEOS_DIR, f"{entry['sequence']}_trex_vis{range_suffix}{suffix}.mp4")
    if os.path.isfile(out_path):
        print(f"[skip] {entry['sequence']}: video already exists")
        return out_path
    cmd = [
        sys.executable, os.path.join(TREX_REPO, "visualize_trex_video_and_map.py"),
        "--video", entry["videos"][0],
        "--tracking-dir", NPZ_DIR,
        "--tracks-csv", os.path.join(entry_out_dir(entry), "tracks_w", "tracks.csv"),
        "--poses", os.path.join(frames_dir_for(entry), "poses.json"),
        "--dem-json", entry["dem_json"],
        "--output", out_path,
        # No --calib here: --video is the original raw video, and --calib would
        # undistort the keypoints into a space the raw video isn't in, misaligning
        # them. Only pass --calib when --video is the undistorted extracted frames.
        # Fish-school outlines/holes: --fschool-dir draws the raw (pixel-space) ones
        # on the video panel; --fschool-geor-dir (Step 3c's output) also draws the
        # geo-referenced ones on the map panel. Both are safe to pass even when a
        # sequence has no fish-school tracklets or Step 3c hasn't run yet - they just
        # resolve to no files and nothing gets drawn.
        "--fschool-dir", NPZ_DIR,
        "--fschool-geor-dir", os.path.join(entry_out_dir(entry), "fschool_georeferenced_w"),
        "--no-live",
    ]
    if RENDER_START_FRAME:
        cmd += ["--start-frame", str(RENDER_START_FRAME)]
    if RENDER_END_FRAME is not None:
        cmd += ["--end-frame", str(RENDER_END_FRAME)]
    elif RENDER_MAX_FRAMES is not None:
        cmd += ["--max-frames", str(RENDER_MAX_FRAMES)]
    if DEBUG_FRAME_LIMIT is not None:
        cmd += ["--max-frames", str(DEBUG_FRAME_LIMIT)]
    print(f"[run]  {entry['sequence']}: rendering georeferenced video"
          + (f" (debug: first {DEBUG_FRAME_LIMIT} frames)" if DEBUG_FRAME_LIMIT is not None else ""))
    run_capped(cmd)
    return out_path

## Run — process one or more sequences end-to-end

Every step above only *defines* its function now - nothing runs until this cell calls
them, in order, for each sequence in `SEQUENCES_TO_RUN`. This replaced the old
"loop every step over all of `manifest`" structure: previously, running Step 2's cell
extracted frames for *all* 11 sequences before you could even start on Step 3, and the
same for every other step. Now each sequence runs its whole chain start-to-finish
before the next one starts, and `SEQUENCES_TO_RUN` controls how many/which:

- an **int N** - the first N *ready* sequences from the manifest, in manifest order
  (e.g. `1` to try a single trial, matching the old single-sequence behaviour);
- a **list of sequence names** - exactly those, in the given order;
- **`None`** - every ready sequence (the original batch-everything behaviour).

Every stage is still individually idempotent (skips work that's already done), so
re-running this cell after an error, or after changing something upstream (e.g. a new
`RENDER_START_FRAME`), only redoes what's actually missing or stale.

In [ ]:
# Options:
#  - an int N: run the first N *ready* sequences from the manifest, in manifest order
#  - a list of sequence names: run exactly those, in the given order
#  - None: run every ready sequence
SEQUENCES_TO_RUN = 11


def resolve_sequences_to_run(spec):
    ready = [e["sequence"] for e in manifest if e["ready"]]
    if spec is None:
        return ready
    if isinstance(spec, int):
        return ready[:spec]
    return list(spec)


def run_pipeline_for_sequence(sequence_name):
    entry = next((e for e in manifest if e["sequence"] == sequence_name), None)
    if entry is None:
        print(f"[skip] {sequence_name}: not in manifest")
        return None
    if not entry["ready"]:
        print(f"[skip] {sequence_name}: not ready ({entry['issues']})")
        return None

    print(f"=== {sequence_name} ===")

    print("-- Step 1: flat-surface DEM --")
    entry["dem_json"] = generate_dem(entry)

    print("-- Step 2: extract frames + poses --")
    extract_frames(entry)
    if not frames_ready(entry):
        print(f"[stop] {sequence_name}: frame extraction did not complete")
        return entry

    print("-- Step 3: geo-reference tracklets (bounding boxes) --")
    run_trex_connector(entry)

    print("-- Step 3b: geo-reference tracklets (per key-point) --")
    export_georeferenced_tracklets(entry)

    print("-- Step 3c: geo-reference fish-school outlines/holes --")
    georeference_fschool(entry)

    print("-- Step 4: render vis video --")
    render_georeferenced_video(entry)

    print(f"=== {sequence_name}: done ===")
    return entry


for _seq in resolve_sequences_to_run(SEQUENCES_TO_RUN):
    run_pipeline_for_sequence(_seq)

=== sequence_20240303_070126703_DJI_0257 ===
-- Step 1: flat-surface DEM --
[skip] sequence_20240303_070126703_DJI_0257: DEM already exists
-- Step 2: extract frames + poses --
[skip] sequence_20240303_070126703_DJI_0257: frames already extracted (75104 frames)
-- Step 3: geo-reference tracklets (bounding boxes) --
[skip] sequence_20240303_070126703_DJI_0257: already geo-referenced
-- Step 3b: geo-reference tracklets (per key-point) --
[skip] sequence_20240303_070126703_DJI_0257: geo-referenced tracklets already exported
-- Step 3c: geo-reference fish-school outlines/holes --
[skip] sequence_20240303_070126703_DJI_0257: fish-school outlines already geo-referenced
-- Step 4: render vis video --
[skip] sequence_20240303_070126703_DJI_0257: video already exists
=== sequence_20240303_070126703_DJI_0257: done ===
=== sequence_20240303_060726300_DJI_0242 ===
-- Step 1: flat-surface DEM --
[skip] sequence_20240303_060726300_DJI_0242: DEM already exists
-- Step 2: extract frames + poses --
[sk

## Step 5 (one-off) — Render a drone-motion-corrected ("warped") video

Step 4's video is pixel-space: the camera pans/rotates with the drone, so the water/seabed
appears to move even when nothing in the scene actually did. This cell instead renders a
single sequence/frame-range through `render_warped_video.py`, which:

1. Projects the 4 corners of each (undistorted) frame onto the same flat-surface/DEM plane
   `trex_to_bambi.py` uses, via one ray-cast per frame (cheap, no video decode needed).
2. Detects the ArUco land markers per frame and measures how much each one still drifts on
   the canvas after the telemetry-only homography above - even smoothed telemetry has
   residual pose error, and a marker on land isn't exactly on the assumed sea-level plane, so
   it shows parallax as the camera moves. Each marker's median canvas position is taken as
   its reference, and every frame gets a corrective transform nudging its marker(s) back onto
   that reference (interpolated across frames with no detection). Verified on real footage:
   this took one marker's canvas-position std from (7.0, 2.1) px down to (0.14, 0.06) px.
3. Derives a 3x3 homography per frame from the corner correspondences, composes the ArUco
   correction into it, and warps the frame with `cv2.warpPerspective` onto a **fixed**
   geographic canvas (+ satellite backdrop) - so the world stays anchored and only the
   camera's footprint window moves around the canvas as the drone moves.
4. Overlays the already geo-referenced `tracks.csv` boxes/trails and the drone's own
   trajectory on top (both already in the same world CRS, so no extra projection needed).

Note on the fish school as a stabilization reference: it deliberately is **not** used for
this - the school deforms and repositions itself independent of the camera, so anchoring on
it would make static things (the seafloor, the markers) drift *more*, not less, since the
school's own motion would get attributed to camera motion instead.

This is a lighter-weight alternative to bambi's full moderngl orthomosaic pipeline - no GPU/
EGL context required, consistent with the rest of this pipeline's "no QGIS needed" approach.
Unlike Steps 1-4 this isn't looped over `manifest` - it's a one-off render for whichever
sequence/frame-range you set below. The ArUco detection pass adds a full extra video
decode+undistort pass (roughly doubles render time) - set `WARP_ARUCO_CORRECT = False` to
skip it if you just want a quick preview.

In [19]:
WARP_SEQUENCE = "sequence_20240303_070126703_DJI_0257"
WARP_START_FRAME = 5000
WARP_END_FRAME = 7000
WARP_CANVAS_SIZE = 1400
# Raw GPS/barometer altitude is sampled far more coarsely than the video frame rate,
# then interpolated as a short, sharp ramp between samples (e.g. a real jump held flat
# for ~100-300 frames, then ramped in over just ~10 frames). Fed straight into a fresh
# per-frame homography, each ramp rescales the whole warped footprint within a fraction
# of a second, which reads as visible trembling even though the flight itself is smooth.
# This smooths the camera trajectory used for the warp only (not the drone-marker
# overlay, and not trex_to_bambi.py's own geo-referencing, which must stay exact).
# Set to 0 or 1 to disable and use the raw poses.
WARP_POSE_SMOOTH_WINDOW = 25
# Detects the ArUco land markers and corrects residual drift against them (see the
# markdown above) - set to False for a quicker preview (skips a full extra video
# decode+undistort pass, roughly doubling render time when enabled).
WARP_ARUCO_CORRECT = True
WARP_ARUCO_DICT = "DICT_4X4_50"  # confirmed working on this survey's markers

WARP_SCRIPT = os.path.join(TREX_REPO, "render_warped_video.py")


def render_warped_video(entry, start_frame, end_frame, canvas_size=WARP_CANVAS_SIZE,
                        pose_smooth_window=WARP_POSE_SMOOTH_WINDOW,
                        aruco_correct=WARP_ARUCO_CORRECT, aruco_dict=WARP_ARUCO_DICT):
    # Encode the smoothing window + aruco-correction setting in the filename too (not
    # just the frame range) so re-running with different settings doesn't wrongly skip
    # against a file rendered with a different one.
    smooth_suffix = f"_sw{pose_smooth_window}" if pose_smooth_window > 1 else ""
    aruco_suffix = "_aruco" if aruco_correct else ""
    out_path = os.path.join(
        GEOREF_VIDEOS_DIR,
        f"{entry['sequence']}_warped_f{start_frame}-{end_frame}{smooth_suffix}{aruco_suffix}.mp4")
    if os.path.isfile(out_path):
        print(f"[skip] {entry['sequence']}: warped video already exists")
        return out_path
    frames_dir = frames_dir_for(entry)
    cmd = [
        TREX_PYTHON, WARP_SCRIPT,
        "--video", entry["videos"][0],
        "--dem-json", entry["dem_json"],
        "--poses", os.path.join(frames_dir, "poses.json"),
        "--calib", entry["calib_path"],
        "--mask", os.path.join(frames_dir, f"mask_{CAMERA}.png"),
        "--tracks-csv", os.path.join(entry_out_dir(entry), "tracks_w", "tracks.csv"),
        "--start-frame", str(start_frame),
        "--end-frame", str(end_frame),
        "--canvas-size", str(canvas_size),
        "--pose-smooth-window", str(pose_smooth_window),
        "--aruco-dict", aruco_dict,
        "--output", out_path,
        "--flat-surface-msl", str(FLAT_SURFACE_MSL),
    ]
    if not aruco_correct:
        cmd.append("--no-aruco-correct")
    print(f"[run]  {entry['sequence']}: rendering warped video (frames {start_frame}-{end_frame})")
    run_capped(cmd)
    return out_path


warp_entry = next((e for e in manifest if e["sequence"] == WARP_SEQUENCE), None)
if warp_entry is None:
    print(f"[skip] {WARP_SEQUENCE}: not in manifest")
elif not frames_ready(warp_entry):
    print(f"[skip] {WARP_SEQUENCE}: frames not extracted yet (step 2 hasn't reached it)")
else:
    render_warped_video(warp_entry, WARP_START_FRAME, WARP_END_FRAME)

[run]  sequence_20240303_070126703_DJI_0257: rendering warped video (frames 5000-7000)
1. Loading DEM + poses
   DEM CRS EPSG:32643, origin offset (257418.02208781848, 430070.75962224265, 0.0)
   Flat-surface projection: 0.0 m MSL (= 0.00 m local). DEM mesh skipped.
   Video: /media/maldives/2024_sequences/sequences-yolo26/sequence_20240303_070126703_DJI_0257.mp4  (5120x2700 @ 50.0 fps, 75104 frames)
   Undistorting (5120, 2700) -> (2700, 2700) using blue_drone_combined.json
   Frame range: 5000 -> 7000 (2000 frames)
2. Projecting frame corners onto the ground plane (pass 1)
   Smoothing camera trajectory over a 25-frame window
   Canvas extent E[257426.0,257435.8] N[430063.7,430073.5]
3. Detecting ArUco markers (DICT_4X4_50) for drift correction (pass 2)
   ArUco: detected in 1987/2000 frames (ids seen: [0, 3, 17])
   marker 0: reference canvas position (330.5, 952.1), seen in 1987 frames
   marker 3: reference canvas position (330.1, 1094.1), seen in 565 frames
   marker 17: referenc